<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms---W7/blob/main/W7D1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="phantoms_documents"
)

documents = [
    "Python lists are mutable sequences that can store items of different types.",
    "Git branches let developers work on features without affecting the main code.",
    "A REST API exposes resources through HTTP methods like GET and POST.",
    "SQL JOINs combine rows from two tables based on a related column.",
    "Regular walking for thirty minutes a day improves heart health.",
    "Drinking enough water helps maintain energy and concentration.",
    "Sleep of seven to eight hours supports muscle recovery and memory.",
    "Stretching before exercise can reduce the risk of muscle injuries.",
    "To make rice fluffy, rinse it first and let it steam after cooking.",
    "Letting dough rest allows the gluten to relax and makes bread softer.",
    "Searing meat at high heat creates a flavorful brown crust.",
    "Adding salt to pasta water seasons the noodles from the inside."
]

ids = [f"doc_{i:02d}" for i in range(1, 13)]

topics = [
    "programming", "programming", "programming", "programming",
    "fitness", "fitness", "fitness", "fitness",
    "cooking", "cooking", "cooking", "cooking"
]

metadatas = [{"topic": topic} for topic in topics]

# Add or update documents without duplication
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Documents inserted successfully!")
print("Total documents:", collection.count())


# Queries
queries = {
    "Q1": "How do I store different kinds of values together in Python?",
    "Q2": "How can I avoid hurting myself when working out?",
    "Q3": "Why is my bread dense and hard?",
    "Q4": "What is the best way to merge data from two database tables?",
    "Q5": "How do I deploy a rocket to Mars?"
}


# Search function
def search_query(query, where=None):

    results = collection.query(
        query_texts=[query],
        n_results=3,
        where=where,
        include=["documents", "metadatas", "distances"]
    )

    for i in range(len(results["documents"][0])):

        print(f"\nResult {i + 1}")
        print("Document:", results["documents"][0][i])
        print("Topic:", results["metadatas"][0][i]["topic"])
        print("Distance:", results["distances"][0][i])

    return results


# Run all five queries
print("\n========== ALL QUERIES ==========")

all_results = {}

for q_id, question in queries.items():

    print(f"\n{q_id}: {question}")

    all_results[q_id] = search_query(question)


# Analyze distances
print("\n========== Q5 THRESHOLD ANALYSIS ==========")

correct_distances = []

for q_id in ["Q1", "Q2", "Q3", "Q4"]:

    distance = all_results[q_id]["distances"][0][0]
    correct_distances.append(distance)

q5_distances = all_results["Q5"]["distances"][0]

max_correct = max(correct_distances)
min_q5 = min(q5_distances)

print("Correct query nearest distances:", correct_distances)
print("Q5 distances:", q5_distances)

if min_q5 > max_correct:

    threshold = (max_correct + min_q5) / 2

    print("Suggested threshold:", threshold)

    if q5_distances[0] > threshold:
        print("Q5: No suitable answer found.")
    else:
        print("Q5: A result passed the threshold.")

else:

    threshold = None

    print("No clear distance gap was found.")
    print("A reliable threshold cannot be determined from these samples.")


# Metadata filtering experiment
print("\n========== Q2 WITHOUT FILTER ==========")

q2_without_filter = collection.query(
    query_texts=[queries["Q2"]],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i in range(3):
    print("\nDocument:", q2_without_filter["documents"][0][i])
    print("Topic:", q2_without_filter["metadatas"][0][i]["topic"])
    print("Distance:", q2_without_filter["distances"][0][i])


print("\n========== Q2 WITH FITNESS FILTER ==========")

q2_with_filter = collection.query(
    query_texts=[queries["Q2"]],
    n_results=3,
    where={"topic": "fitness"},
    include=["documents", "metadatas", "distances"]
)

for i in range(len(q2_with_filter["documents"][0])):
    print("\nDocument:", q2_with_filter["documents"][0][i])
    print("Topic:", q2_with_filter["metadatas"][0][i]["topic"])
    print("Distance:", q2_with_filter["distances"][0][i])


# Save results to notes.md
notes = []

notes.append("# W7D1 - Vector Database with ChromaDB\n")

notes.append("## 1. Query Results\n")

notes.append(
    "| Query | First Result Correct? | Returned Document |\n"
    "|---|---|---|\n"
)

expected_topics = {
    "Q1": "programming",
    "Q2": "fitness",
    "Q3": "cooking",
    "Q4": "programming"
}

for q_id in ["Q1", "Q2", "Q3", "Q4", "Q5"]:

    result = all_results[q_id]

    first_document = result["documents"][0][0]
    first_topic = result["metadatas"][0][0]["topic"]

    if q_id == "Q5":
        correctness = "No suitable answer expected"
    else:
        correctness = (
            "Yes" if first_topic == expected_topics[q_id] else "No"
        )

    notes.append(
        f"| {q_id} | {correctness} | {first_document} |\n"
    )


notes.append("\n## 2. Q5 Distance Analysis\n")
notes.append(f"- Correct queries nearest distances: {correct_distances}\n")
notes.append(f"- Q5 distances: {q5_distances}\n")
notes.append(f"- Suggested threshold: {threshold}\n")

if threshold is not None:
    notes.append(
        "- Rule: Reject a query when its nearest result distance "
        "is greater than the suggested threshold.\n"
    )
else:
    notes.append(
        "- No reliable threshold could be determined because "
        "the observed distances overlap.\n"
    )

notes.append("\n## 3. Metadata Filtering\n")
notes.append("Q2 without filter:\n")

for doc in q2_without_filter["documents"][0]:
    notes.append(f"- {doc}\n")

notes.append("\nQ2 with fitness filter:\n")

for doc in q2_with_filter["documents"][0]:
    notes.append(f"- {doc}\n")


notes.append("\n## 4. Duplicate Prevention\n")
notes.append(
    f"- Used upsert() with fixed IDs.\n"
    f"- Total documents after insertion: {collection.count()}\n"
    f"- Expected count: 12\n"
)

notes.append("\n## 5. Default Distance Metric\n")
notes.append(
    "- ChromaDB default distance metric is squared L2 (l2).\n"
    "- Documentation: https://docs.trychroma.com/docs/collections/configure\n"
)

notes.append("\n## 6. Vector DB vs SQL\n")
notes.append(
    "- SQL databases organize structured data into tables "
    "and support exact conditions and relational queries.\n"
)
notes.append(
    "- Vector databases store embeddings and retrieve "
    "documents based on semantic similarity.\n"
)

notes.append("\n## 7. Why Q5 Matters in RAG\n")
notes.append(
    "- Vector search always tries to retrieve the nearest available "
    "documents, even when they are unrelated to the question.\n"
)
notes.append(
    "- This can cause a RAG system to provide irrelevant context "
    "and generate incorrect answers.\n"
)
notes.append(
    "- A distance threshold can help reject unrelated results, "
    "but it must be evaluated and calibrated using real data.\n"
)

with open("notes.md", "w", encoding="utf-8") as file:
    file.writelines(notes)

print("\nnotes.md created successfully!")
print("\n========== FINAL CHECK ==========")
print("Total documents:", collection.count())
print("Expected documents: 12")
print("Task completed!")